# Study 949 — Riding the TIPS Curve 🪜

**Is there a roll-down carry in real yields, or only duration risk?**

The pitch is old and tidy. The *real* yield curve slopes upward, so an inflation-linked
bond held for a year ages down that curve and gets repriced at a lower real yield — a
capital gain on top of the running real coupon. Extend from short linkers into long ones,
the story goes, and you pick up the roll.

We test both halves of that claim on **VTIP / SCHP / TIP / LTPZ vs BIL** (cash), daily
**total-return** closes, 2012-10-16 → 2026-06-30 (3,444 days):

1. **Does extending pay?** Race the four maturity buckets **excess-of-cash** — minus
   BIL's actual total return, not a flat proxy.
2. **Is what is left a carry?** Strip the duration out of each linker by shorting a
   duration-matched **nominal** Treasury fund, with the hedge beta fitted through day
   *t*−1 and applied at *t*, and put a Newey-West *t* on the residual.

*Every real number below is the frozen headline (`docs/results.md`, Fingerprint
`9da861d9d566`); the live cells run the offline synthetic control only. As-of 2026-06-30.*


## 1. First, the simple question: did going longer pay?

Four funds, same asset class, different maturities. VTIP holds linkers with under five years to run; LTPZ holds the fifteen-year-plus end. If riding the real curve pays, the long one should have earned more over cash than the short one. Here is what thirteen and a half years actually delivered.

In [1]:
L = {'VTIP': {'dur': '~2.5y', 'exret': 0.65, 'vol': 2.54, 'sharpe': 0.256, 't': 0.99, 'exdd': -7.4, 'cagr': 2.23, 'absdd': -6.3, 'ci': (-0.193, 0.767), 'neg': 13.6}, 'SCHP': {'dur': '~7y', 'exret': 0.35, 'vol': 5.48, 'sharpe': 0.064, 't': 0.24, 'exdd': -18.4, 'cagr': 1.8, 'absdd': -14.3, 'ci': (-0.409, 0.575), 'neg': 38.8}, 'TIP': {'dur': '~7y', 'exret': 0.27, 'vol': 5.61, 'sharpe': 0.049, 't': 0.19, 'exdd': -18.7, 'cagr': 1.71, 'absdd': -14.5, 'ci': (-0.417, 0.545), 'neg': 41.8}, 'LTPZ': {'dur': '~20y', 'exret': -0.02, 'vol': 14.68, 'sharpe': -0.001, 't': -0.0, 'exdd': -44.8, 'cagr': 0.49, 'absdd': -41.0, 'ci': (-0.462, 0.454), 'neg': 49.8}}
CASH_CAGR = 1.59
print(f"{'bucket':6s} {'duration':9s} {'over cash':>10s} {'vol':>7s} "
      f"{'exSharpe':>9s} {'worst DD':>9s}")
for k in ('VTIP','SCHP','TIP','LTPZ'):
    d = L[k]
    print(f"{k:6s} {d['dur']:9s} {d['exret']:+9.2f}% {d['vol']:6.2f}% "
          f"{d['sharpe']:+9.3f} {d['absdd']:8.1f}%")
print(f"\ncash (BIL) itself compounded at {CASH_CAGR:+.2f}%/yr over the same window")

bucket duration   over cash     vol  exSharpe  worst DD
VTIP   ~2.5y         +0.65%   2.54%    +0.256     -6.3%
SCHP   ~7y           +0.35%   5.48%    +0.064    -14.3%
TIP    ~7y           +0.27%   5.61%    +0.049    -14.5%
LTPZ   ~20y          -0.02%  14.68%    -0.001    -41.0%

cash (BIL) itself compounded at +1.59%/yr over the same window


## 2. The uncomfortable answer

Going longer did not pay more. It paid **less**, all the way down the ladder — and the longest bucket did not beat cash at all. Over thirteen and a half years, long TIPS (LTPZ) compounded at **+0.49%/yr** while the T-bill leg paid **+1.59%/yr** for doing nothing — and it cost a **-41%** drawdown to find that out. Short linkers (VTIP) were the only bucket to clear cash, at a tenth of the volatility.

To be scrupulous: none of these gaps is statistically significant. The honest sentence is not *"long linkers lose"* — it is **"nothing in this ladder can be told apart from zero"**. Real duration went unrewarded.

> 🔬 **For the quants.** Every leg is measured excess of BIL's *realised* total return, so the race spans the zero-rate decade and the 5% regime without a flat-rate fudge. Long-minus-short HAC *t*s: SCHP−VTIP -0.32, TIP−VTIP -0.39, LTPZ−VTIP -0.21. Bootstrap Sharpe CIs straddle zero for all four buckets.

## 3. So where did the roll-down go?

Maybe the carry is there but buried under duration — the funds all rise and fall with the level of rates, and 2022 alone swamped a decade of coupon. So we take the duration out: hold the linker, short just enough of a **maturity-matched ordinary Treasury fund** to cancel the rate exposure, and look at what is left. That residual is the closest thing to a pure *inflation-linked-versus-nominal* carry you can build from listed funds.

In [2]:
S = {'VTIP-SHY': {'beta': 1.12, 'gross': 0.98, 'tg': 1.55, 'net': 0.6, 'tn': 0.94, 'sharpe': 0.277, 'vol': 2.16, 'turn': 2.3}, 'SCHP-IEF': {'beta': 0.68, 'gross': 0.69, 'tg': 0.68, 'net': 0.47, 'tn': 0.47, 'sharpe': 0.141, 'vol': 3.35, 'turn': 0.6}, 'TIP-IEF': {'beta': 0.69, 'gross': 0.55, 'tg': 0.54, 'net': 0.33, 'tn': 0.33, 'sharpe': 0.096, 'vol': 3.49, 'turn': 0.6}, 'LTPZ-TLT': {'beta': 0.86, 'gross': 0.34, 'tg': 0.15, 'net': 0.07, 'tn': 0.03, 'sharpe': 0.009, 'vol': 8.22, 'turn': 0.6}}
for k in ('VTIP-SHY','SCHP-IEF','TIP-IEF','LTPZ-TLT'):
    d = S[k]
    print(f"{k:10s} residual carry {d['gross']:+.2f}%/yr gross "
          f"({d['net']:+.2f}%/yr after costs and borrow)   t = {d['tg']:+.2f}")
print('\nA t-statistic needs to reach about 2 before this desk calls anything real.')

VTIP-SHY   residual carry +0.98%/yr gross (+0.60%/yr after costs and borrow)   t = +1.55
SCHP-IEF   residual carry +0.69%/yr gross (+0.47%/yr after costs and borrow)   t = +0.68
TIP-IEF    residual carry +0.55%/yr gross (+0.33%/yr after costs and borrow)   t = +0.54
LTPZ-TLT   residual carry +0.34%/yr gross (+0.07%/yr after costs and borrow)   t = +0.15

A t-statistic needs to reach about 2 before this desk calls anything real.


## 4. Positive everywhere, provable nowhere

Something *is* there — the residual is positive in all four buckets. But the biggest *t* in the whole study is **+1.55**, and it is the **shortest** bucket, not the longest. That is backwards for a roll-down story: if the slope of the real curve were what you were being paid for, the carry should grow as you extend. Instead it shrinks — +0.98%/yr at the front end, +0.34%/yr at the long end.

## 5. And then you look at *when* it happened

In [3]:
Y = {2013: -0.47, 2014: -2.36, 2015: -0.54, 2016: 1.54, 2017: 0.55, 2018: -0.86, 2019: 1.53, 2020: 1.79, 2021: 6.38, 2022: 1.63, 2023: 0.51, 2024: 0.63, 2025: 1.19, 2026: 0.87}
for y, v in Y.items():
    bar = '#' * max(0, int(round(v * 6)))
    tag = '   <-- the inflation shock lands' if y == 2021 else ''
    print(f"{y}  {v:+6.2f}%  {bar}{tag}")

2013   -0.47%  
2014   -2.36%  
2015   -0.54%  
2016   +1.54%  #########
2017   +0.55%  ###
2018   -0.86%  
2019   +1.53%  #########
2020   +1.79%  ###########
2021   +6.38%  ######################################   <-- the inflation shock lands
2022   +1.63%  ##########
2023   +0.51%  ###
2024   +0.63%  ####
2025   +1.19%  #######
2026   +0.87%  #####


## 6. One year is the whole story

**2021 alone made +6.38%** — four times the full-sample average — and four of the eight years before it were *negative* (2013 is a 50-day part year, so treat that count as a tally of rows, not of full years). Cut the sample around the 2021-2023 inflation shock and the pattern is unmissable: the short sleeve earned **-0.20%/yr net across the 7.2 pre-shock years**, **+2.36%/yr during 2021-2023**, and **+0.79%/yr since**. Remove 2021 and the whole result collapses to **+0.17%/yr** (*t* = +0.26).

There is a reason, and it is not roll-down. Owning a linker and shorting an ordinary bond is, mechanically, a bet that **inflation comes in higher than the market had priced**. In 2021-2023 it did, spectacularly. That leg had to pay, whatever the real curve's slope was doing — and a roll-down carry does not switch itself on when CPI surprises.

> 🔬 **For the quants.** Roll-down in real yields and the realised-minus-expected inflation term are **not separately identifiable** from fund total returns; the residual is a long-breakeven position by construction. That unidentifiability is precisely why the era cut, not the full-sample point estimate, decides the verdict here. And note where the study's *only* |*t*| ≥ 2 lives: the **gross** carry inside that shock window, *t* = **+2.04** — one hit out of 56 tests, on a sub-window chosen after the fact. It is the rival hypothesis arriving on schedule, not the claim surviving.

## 7. Could you trade it anyway?

The residual is a *spread*: long the linker, short the Treasury fund, every day. Shorting is not free — you pay a borrow fee on the short leg for as long as you hold it. Watch what a perfectly ordinary borrow rate does to the best number in the study.

In [4]:
B = [(0.0, 0.93, 1.47, 0.433), (25.0, 0.65, 1.03, 0.303), (30.0, 0.6, 0.94, 0.277), (50.0, 0.37, 0.59, 0.173), (100.0, -0.19, -0.29, -0.087)]
for bo, net, t, sh in B:
    note = '   <- free shorting: not a thing' if bo == 0 else ''
    note = '   <- base case' if bo == 30 else note
    note = '   <- edge gone' if net < 0 else note
    print(f"borrow {bo:5.0f} bps/yr  ->  net {net:+.2f}%/yr  (t {t:+.2f}, "
          f"Sharpe {sh:+.3f}){note}")

borrow     0 bps/yr  ->  net +0.93%/yr  (t +1.47, Sharpe +0.433)   <- free shorting: not a thing
borrow    25 bps/yr  ->  net +0.65%/yr  (t +1.03, Sharpe +0.303)
borrow    30 bps/yr  ->  net +0.60%/yr  (t +0.94, Sharpe +0.277)   <- base case
borrow    50 bps/yr  ->  net +0.37%/yr  (t +0.59, Sharpe +0.173)
borrow   100 bps/yr  ->  net -0.19%/yr  (t -0.29, Sharpe -0.087)   <- edge gone


## 8. Is the measuring stick broken? (live, offline synthetic)

Before accepting a null, check the instrument. We build an artificial world where a **2%/yr residual carry really exists**, hand it to the same code, and see whether it finds it — then repeat with a world where there is nothing planted at all. Nothing here touches the real tape.

In [5]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from tips_roll import data, strategy as st
planted = st.synthetic_detect(data.synthetic_daily(signal_strength=1.0, seed=949)[0])
null    = st.synthetic_detect(data.synthetic_daily(signal_strength=0.0, seed=949)[0])
print('world with a real 2.00%%/yr carry : found %+.2f%%/yr, t = %+.2f'
      % (planted['gross_ann']*100, planted['t_gross']))
print('world with no carry at all       : found %+.2f%%/yr, t = %+.2f'
      % (null['gross_ann']*100, null['t_gross']))

world with a real 2.00%/yr carry : found +2.19%/yr, t = +3.65
world with no carry at all       : found +0.19%/yr, t = +0.32


The detector finds a genuine 2%/yr carry at *t* well past 3, on a **shorter** sample than the real one, and stays quiet when there is nothing to find. The instrument works. The real tape simply has no roll-down carry in it.

## Verdict

- **Signal — None.** Extending along the real curve paid nothing (excess Sharpe falls from **+0.256** at the front to **-0.001** at the long end, every difference insignificant). The duration-hedged residual is positive in all four buckets and significant in none — best **full-sample** *t* **+1.55** gross, **+0.94** net — it *shrinks* as you extend, and it is one inflation shock wearing a carry costume. Of the **56** *t*s this study computes on the real tape, **1** reaches 2, and it is the gross carry inside the shock window itself (Bonferroni bar ≈ 3.2).
- **Tradability — Mirage.** The best net figure anywhere is **+0.84%/yr** (504-day beta, *t* = +1.24); the base case pays **+0.60%/yr** — both from a sleeve short a Treasury fund every day; it dies at about **1%/yr borrow**, and it dies outright if you remove a single calendar year. Buying the long bucket as a fund instead was worse: **+0.49%/yr** against cash's **+1.59%**, for a **-41%** drawdown.
- **What the tape does say.** The least-bad place on the linker curve was the **front**. That is a statement about where risk went *unrewarded*, not an edge you can harvest.